# Quran exam results in Jeddah: cleaning and EDA

Exam records from a Quran memorization charity in Jeddah, coming from two exports of the old system (`old_system_old.xls` and `old_system_new.xls`).

**What a row means:** one student sitting one exam. Main fields: student, exam branch, degree and grade, teacher, halaqa (the circle the teacher runs, inside a mosque), mosque, sector (an area of Jeddah such as north or south) and the sector manager.

**Exam branches:** the Quran has 30 juz, but exams are only held at these branches: 1, 2, 3, 5, 8, 10, 13, 15, 20, 25, 30. Anything else in the branch column is treated as a data entry problem.

Dates in both systems are Hijri, so they are converted to Gregorian first. The two systems write them differently (`1425/02/02` vs `21-04-1423`).

### Questions, and what I expect before looking

1. Is the number of exams growing year over year, or did it plateau?
2. Do the later branches (20, 25, 30) have lower pass rates? My guess is yes, but only slightly, since students who reach them are already strong.
3. Do sectors differ in pass rate, or does the difference just come from which branches they happen to test?
4. Are there mosques or teachers with unusually good or bad results, or is it mostly small-sample noise?
5. Do students who fail a branch come back and pass it?
6. Is there seasonality (Ramadan, end of the school year, weekdays)?
7. Did COVID reduce the number of exams, and did it change results once branch mix is taken into account?

The last cells of the notebook print the answers from the data, so nothing in the text above is a conclusion.

In [ ]:
import re, csv, warnings
from pathlib import Path
from html.parser import HTMLParser

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

# Arabic text in matplotlib needs reshaping. If these two packages are missing
# the plots still work, the Arabic labels just look disconnected.
#   pip install arabic-reshaper python-bidi hijridate
try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    def ar(x):
        return get_display(arabic_reshaper.reshape(str(x)))
except ImportError:
    def ar(x):
        return str(x)

try:
    from hijridate import Hijri
except ImportError:
    from hijri_converter import Hijri   # older package name

In [ ]:
# ---- settings (change these, not the code below) ----
DATA_DIR = Path('.')
OLD_XLS = DATA_DIR / 'old_system_old.xls'
NEW_XLS = DATA_DIR / 'old_system_new.xls'

VALID_BRANCHES = [1, 2, 3, 5, 8, 10, 13, 15, 20, 25, 30]
ANALYSIS_START, ANALYSIS_END = 2010, 2025     # years kept for the analysis
MIN_EXAMS = 100                               # minimum exams before ranking a teacher / mosque / manager

# COVID windows (Gregorian). Suspension in Saudi Arabia started in March 2020 and
# in-person education came back around September 2021. Adjust if the charity's
# own calendar was different.
COMPARE_FROM = pd.Timestamp('2017-01-01')
COVID_START  = pd.Timestamp('2020-03-01')
COVID_END    = pd.Timestamp('2021-09-01')

GRADE_ORDER = ['Excellent', 'Very Good', 'Good', 'Acceptable', 'Fail']
GRADE_COLORS = ['#2a9d8f', '#8ab17d', '#e9c46a', '#f4a261', '#e76f51']

## 1. Loading

The `.xls` files are really HTML tables, so `read_excel` cannot open them. I turn each one into a CSV once and reuse it on later runs.

In [ ]:
class TableToCSV(HTMLParser):
    """Walk through the html table and write every <tr> as one csv row."""

    def __init__(self, path):
        super().__init__()
        self.f = open(path, 'w', encoding='utf-8-sig', newline='')
        self.writer = csv.writer(self.f)
        self.row, self.cell, self.in_cell = [], [], False

    def handle_starttag(self, tag, attrs):
        if tag == 'tr':
            self.row = []
        elif tag in ('td', 'th'):
            self.in_cell, self.cell = True, []

    def handle_endtag(self, tag):
        if tag in ('td', 'th'):
            self.in_cell = False
            self.row.append(''.join(self.cell).strip())
        elif tag == 'tr' and self.row:
            self.writer.writerow(self.row)

    def handle_data(self, data):
        if self.in_cell:
            self.cell.append(data)

    def close(self):
        super().close()
        self.f.close()


def html_xls_to_csv(src, dst):
    raw = Path(src).read_bytes()
    try:
        text = raw.decode('utf-8')
    except UnicodeDecodeError:
        text = raw.decode('windows-1256', errors='replace')
    parser = TableToCSV(dst)
    parser.feed(text)
    parser.close()


def load_system(xls_path):
    csv_path = Path(xls_path).with_name(Path(xls_path).stem + '_clean.csv')
    if not csv_path.exists():
        print(f'converting {xls_path} ...')
        html_xls_to_csv(xls_path, csv_path)
    out = pd.read_csv(csv_path, low_memory=False)
    out.columns = out.columns.str.strip()
    return out


raw_old = load_system(OLD_XLS)
raw_new = load_system(NEW_XLS)
print('old system:', raw_old.shape, '| new system:', raw_new.shape)

The two exports use different column names and the old one has no sector manager. I rename both to one schema and stack them. `exam_id` is only unique inside each system, so I keep a `data_source` column.

In [ ]:
OLD_MAP = {'ot_id': 'exam_id', 'ot_stid': 'student_id', 'ot_stname': 'student_name',
           'ot_date': 'exam_date_raw', 'ot_branch': 'exam_branch', 'ot_degree': 'exam_degree',
           'ot_takdeer': 'exam_grade_raw', 'ot_teacherid': 'teacher_id', 'ot_teachername': 'teacher_name',
           'ot_msq': 'mosque_name', 'ot_sv': 'supervisor_name', 'ot_ring': 'halaqa_name',
           'ot_section': 'sector_name'}

NEW_MAP = {'ne_id': 'exam_id', 'st_id': 'student_id', 'st_name': 'student_name',
           'examdate': 'exam_date_raw', 'ne_branchno': 'exam_branch', 'ne_degree': 'exam_degree',
           'nr_desc': 'exam_grade_raw', 'tf_id': 'teacher_id', 'teacher': 'teacher_name',
           'msq_name': 'mosque_name', 'supervisor': 'supervisor_name', 'rg_name': 'halaqa_name',
           'sec_name': 'sector_name', 'secmanger': 'sector_manager'}

KEEP = ['exam_id', 'student_id', 'student_name', 'exam_date_raw', 'exam_branch', 'exam_degree',
        'exam_grade_raw', 'teacher_id', 'teacher_name', 'mosque_name', 'supervisor_name',
        'halaqa_name', 'sector_name', 'sector_manager', 'data_source']

old = raw_old.rename(columns=OLD_MAP).assign(data_source='old')
new = raw_new.rename(columns=NEW_MAP).assign(data_source='new')

df = pd.concat([old.reindex(columns=KEEP), new.reindex(columns=KEEP)], ignore_index=True)
print(df.shape)
df.head()

## 2. Cleaning

I keep a small log of every step (rows removed or values changed) so the cleaning can be explained later and nothing disappears silently.

In [ ]:
clean_log = []
def log(step, rows_affected, note=''):
    clean_log.append({'step': step, 'rows_affected': int(rows_affected), 'note': note})

# --- text columns: collapse spaces, blank -> missing -----------------------
text_cols = ['student_name', 'teacher_name', 'mosque_name', 'supervisor_name',
             'halaqa_name', 'sector_name', 'sector_manager', 'exam_grade_raw']

def tidy(v):
    if pd.isna(v):
        return np.nan
    v = re.sub(r'\s+', ' ', str(v)).strip()
    return v if v and v.lower() != 'nan' else np.nan

for c in text_cols:
    df[c] = df[c].map(tidy)

# mosque names carry a "(مجمع قرآني)" suffix in some rows only
before = df['mosque_name'].copy()
df['mosque_name'] = df['mosque_name'].str.replace(r'\s*\(.*?\)', '', regex=True).str.strip()
log('mosque suffix removed', (before != df['mosque_name']).sum(), 'so the same mosque is not split in two')

# --- numbers -----------------------------------------------------------------
for c in ['exam_branch', 'exam_degree', 'student_id', 'teacher_id']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

### Dates

Old system writes `YYYY/MM/DD`, new system writes `DD-MM-YYYY`, both Hijri. I split the parts, find which side holds the year, and convert only the distinct dates (a few thousand) instead of every row, which is much faster.

In [ ]:
parts = df['exam_date_raw'].astype('string').str.strip().str.extract(r'^(\d{1,4})[-/](\d{1,2})[-/](\d{1,4})$')
a, b, c = (pd.to_numeric(parts[i], errors='coerce') for i in range(3))
year_first = a > 31
df['h_year']  = np.where(year_first, a, c)
df['h_month'] = b
df['h_day']   = np.where(year_first, c, a)

def to_gregorian(y, m, d):
    # some exports contain day 30 in a 29-day month, so step back until it works
    for day in (d, d - 1, d - 2):
        try:
            return pd.Timestamp(Hijri(int(y), int(m), int(day)).to_gregorian())
        except Exception:
            continue
    return pd.NaT

uniq = df[['h_year', 'h_month', 'h_day']].dropna().drop_duplicates().copy()
uniq['exam_date'] = [to_gregorian(*r) for r in uniq.itertuples(index=False)]
df = df.merge(uniq, on=['h_year', 'h_month', 'h_day'], how='left')

df['exam_year']  = df['exam_date'].dt.year
df['exam_month'] = df['exam_date'].dt.month

bad_date = df['exam_date'].isna().sum()
log('date could not be parsed or converted', bad_date, 'kept in the table, excluded from time analysis')
print('Date range:', df['exam_date'].min().date(), '->', df['exam_date'].max().date())
print('Unparsed dates:', bad_date)

A look at the year counts before choosing the analysis window. Very old years or a few rows far in the future are usually typing mistakes.

In [ ]:
df['exam_year'].value_counts().sort_index().to_frame('rows').T

### Grades and branches

The grade column holds Arabic words with different spellings (`جيدجداً`, `جيد جداً`...). I strip spaces and diacritics, then map to five levels. Pass means anything other than Fail.

In [ ]:
GRADE_MAP = {'ممتاز': 'Excellent', 'جيدجدا': 'Very Good', 'جيد': 'Good',
             'مقبول': 'Acceptable', 'راسب': 'Fail'}

def norm_grade(v):
    if pd.isna(v):
        return np.nan
    t = re.sub(r'[\s\u064B-\u0652\u0640]', '', str(v))
    t = t.replace('أ', 'ا').replace('إ', 'ا')
    return GRADE_MAP.get(t, 'Other')

lookup = {v: norm_grade(v) for v in df['exam_grade_raw'].dropna().unique()}
df['grade'] = df['exam_grade_raw'].map(lookup)

# check the mapping did what I think it did
pd.crosstab(df['exam_grade_raw'].fillna('(missing)'), df['grade'].fillna('(missing)'))

In [ ]:
df['branch_ok'] = df['exam_branch'].isin(VALID_BRANCHES)
log('branch not in the official list', (~df['branch_ok']).sum(), 'flagged, excluded from branch analysis')
log('grade missing', df['grade'].isna().sum(), 'excluded from pass-rate analysis')
log('grade not recognised', (df['grade'] == 'Other').sum(), 'excluded from pass-rate analysis')

print('Branch values that are not official:')
print(df.loc[~df['branch_ok'], 'exam_branch'].value_counts(dropna=False).head(15))

Before trusting the numeric degree, I want to see how it behaves inside each grade. The old data had many failed exams with a degree of 1, which looks like a placeholder and not a real mark. If that is true, averaging degrees across everyone would be misleading, so I will average over passing students only.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
order = [g for g in GRADE_ORDER if g in df['grade'].unique()]
sns.boxplot(data=df[df['grade'].isin(order)], x='grade', y='exam_degree', order=order,
            palette=dict(zip(GRADE_ORDER, GRADE_COLORS)), ax=ax)
ax.set_title('Degree inside each grade')
ax.set_xlabel('')
plt.show()

df[df['grade'].isin(order)].groupby('grade')['exam_degree'].describe().reindex(order)

### Duplicates

Same student, same branch, same day and same degree is almost certainly the same exam entered twice, or present in both systems. First I check how much the two systems overlap.

In [ ]:
df['student_key'] = df['student_name'].str.replace(r'[\u064B-\u0652\u0640]', '', regex=True)
df['student_key'] = df['student_key'].str.replace('أ|إ|آ', 'ا', regex=True)
# names are the only student field present in nearly every row (ids are missing for many old rows)

dup_cols = ['student_key', 'exam_branch', 'exam_date', 'exam_degree']
has_key = df[dup_cols].notna().all(axis=1)

both = (df[has_key].groupby(dup_cols)['data_source'].nunique() > 1).sum()
print(f'Exams that appear in BOTH systems: {both:,}')

is_dup = df.duplicated(subset=dup_cols, keep='last') & has_key
log('duplicate exams removed', is_dup.sum(), 'same student/branch/date/degree; kept the newer system copy')
print(f'Duplicates removed: {is_dup.sum():,} of {len(df):,}')

df = df[~is_dup].reset_index(drop=True)

### Missing values

What is missing is not equally important. Sector and manager can be recovered from other columns, teacher and grade cannot, so I only fill what has a sensible source.

- **sector**: each mosque belongs to one sector, so use the most common sector of that mosque
- **sector manager**: managers change over time, so use the most common manager of that sector *in the same Hijri year*

In [ ]:
def fill_from_mode(frame, target, keys):
    ref = (frame.dropna(subset=keys + [target]).groupby(keys)[target]
                .agg(lambda s: s.value_counts().index[0]).rename('_fill').reset_index())
    tmp = frame[keys].merge(ref, on=keys, how='left')
    return frame[target].fillna(pd.Series(tmp['_fill'].values, index=frame.index))

missing_before = df[['sector_name', 'sector_manager']].isna().sum()

df['sector_name'] = fill_from_mode(df, 'sector_name', ['mosque_name'])
df['sector_manager'] = fill_from_mode(df, 'sector_manager', ['sector_name', 'h_year'])

filled = missing_before - df[['sector_name', 'sector_manager']].isna().sum()
log('sector filled from mosque', filled['sector_name'])
log('sector manager filled from sector + year', filled['sector_manager'])

miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].to_frame('share_missing').style.format('{:.1%}')

In [ ]:
# analysis tables ----------------------------------------------------------
d = df[df['exam_date'].notna()
       & df['exam_year'].between(ANALYSIS_START, ANALYSIS_END)
       & df['branch_ok']].copy()
log('outside analysis window / bad date / bad branch', len(df) - len(d), f'kept {ANALYSIS_START}-{ANALYSIS_END}')

d['sector_name'] = d['sector_name'].fillna('Unknown')
d['exam_branch'] = d['exam_branch'].astype(int)

# g = rows with a usable grade, used for every pass-rate calculation
g = d[d['grade'].isin(GRADE_ORDER)].copy()
g['passed'] = (g['grade'] != 'Fail').astype(int)
g['is_excellent'] = (g['grade'] == 'Excellent').astype(int)
g['degree_passed'] = g['exam_degree'].where(g['passed'] == 1)

# what a branch is expected to score, used later to compare mosques, teachers etc.
branch_rate = g.groupby('exam_branch')['passed'].mean()
g['expected'] = g['exam_branch'].map(branch_rate)

pd.DataFrame(clean_log)

In [ ]:
print(f'Final: {len(d):,} exams in the window, {len(g):,} with a usable grade, '
      f'{g["student_key"].nunique():,} students, {g["teacher_name"].nunique():,} teachers, '
      f'{g["mosque_name"].nunique():,} mosques.')

def summarize(frame, by, min_n=1):
    t = frame.groupby(by, observed=True).agg(
        exams=('passed', 'size'),
        pass_rate=('passed', 'mean'),
        excellent_rate=('is_excellent', 'mean'),
        avg_degree_passed=('degree_passed', 'mean'))
    return t[t['exams'] >= min_n]

def vs_expected(frame, by, min_n):
    t = frame.groupby(by).agg(exams=('passed', 'size'), actual=('passed', 'mean'),
                              expected=('expected', 'mean'))
    t['gap'] = t['actual'] - t['expected']
    return t[t['exams'] >= min_n].sort_values('gap')

## 3. Volume and results over time

In [ ]:
yr = summarize(g, 'exam_year')

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yr.index, yr['exams'], color='#9ecae1')
ax.set_ylabel('exams')
ax.set_xticks(yr.index)
ax.tick_params(axis='x', rotation=45)
ax2 = ax.twinx()
ax2.plot(yr.index, yr['pass_rate'] * 100, color='#d62728', marker='o')
ax2.set_ylabel('pass rate %')
ax2.grid(False)
ax2.spines['right'].set_visible(True)
ax.set_title('Exams per year (bars) and pass rate (line)')
plt.tight_layout()
plt.show()
yr

In [ ]:
heat = pd.crosstab(g['exam_year'], g['exam_month'])
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heat, cmap='Blues', annot=True, fmt='d', annot_kws={'size': 8}, cbar=False, ax=ax)
ax.set_title('Exams by Gregorian month and year')
ax.set_xlabel('month'); ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 4. Grades

In [ ]:
overall = g['grade'].value_counts().reindex(GRADE_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(overall.index, overall.values, color=GRADE_COLORS)
axes[0].set_title('Grade distribution (all years)')
for i, v in enumerate(overall.values):
    axes[0].text(i, v, f'{v / overall.sum():.1%}', ha='center', va='bottom', fontsize=9)

by_year = pd.crosstab(g['exam_year'], g['grade'], normalize='index')[GRADE_ORDER] * 100
by_year.plot(kind='bar', stacked=True, color=GRADE_COLORS, ax=axes[1], width=0.85)
axes[1].set_title('Grade mix by year (%)')
axes[1].set_xlabel('')
axes[1].legend(ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.12), frameon=False)
plt.tight_layout()
plt.show()

## 5. Exam branches

In [ ]:
br = summarize(g, 'exam_branch').reindex(VALID_BRANCHES)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
sns.barplot(x=br.index, y=br['exams'], color='#6baed6', ax=axes[0])
axes[0].set_title('Exams per branch'); axes[0].set_xlabel('branch (juz)')

sns.barplot(x=br.index, y=br['pass_rate'] * 100, color='#74c476', ax=axes[1])
axes[1].set_title('Pass rate %'); axes[1].set_xlabel('branch (juz)')
lo = max(0, np.floor(br['pass_rate'].min() * 100 - 5))
axes[1].set_ylim(lo, 100)

sns.boxplot(data=g[g['passed'] == 1], x='exam_branch', y='exam_degree', order=VALID_BRANCHES,
            color='#fdae6b', fliersize=1, ax=axes[2])
axes[2].set_title('Degree of students who passed'); axes[2].set_xlabel('branch (juz)')
plt.tight_layout()
plt.show()
br

## 6. Sectors

A sector can look weak just because it tests more of the hard branches. So every comparison below also shows `gap`: actual pass rate minus the rate you would expect from the sector's branch mix. Positive means better than expected.

In [ ]:
sec = vs_expected(g, 'sector_name', MIN_EXAMS)
top_sectors = sec.sort_values('exams', ascending=False).head(7).index

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
sec_year = g[g['sector_name'].isin(top_sectors)].groupby(['exam_year', 'sector_name']).size().reset_index(name='exams')
sec_year['sector_name'] = sec_year['sector_name'].map(ar)
sns.lineplot(data=sec_year, x='exam_year', y='exams', hue='sector_name', marker='o', ax=axes[0])
axes[0].set_title('Exams per year, biggest sectors')
axes[0].legend(title='', fontsize=8)

s = sec.sort_values('gap')
axes[1].barh([ar(i) for i in s.index], s['gap'] * 100,
             color=np.where(s['gap'] >= 0, '#2a9d8f', '#e76f51'))
axes[1].axvline(0, color='k', lw=0.8)
axes[1].set_title('Pass rate vs expected from branch mix (percentage points)')
plt.tight_layout()
plt.show()
sec.sort_values('exams', ascending=False)

In [ ]:
mgr = vs_expected(g.dropna(subset=['sector_manager']), 'sector_manager', MIN_EXAMS)
print(f'Sector managers with at least {MIN_EXAMS} exams: {len(mgr)} '
      f'({g["sector_manager"].isna().mean():.0%} of rows have no manager)')
mgr.sort_values('exams', ascending=False).head(15)

## 7. Mosques, halaqat and teachers

In [ ]:
mos = vs_expected(g, 'mosque_name', MIN_EXAMS)
top = g['mosque_name'].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh([ar(i) for i in top.index][::-1], top.values[::-1], color='#6baed6')
axes[0].set_title('Mosques with the most exams')

ext = pd.concat([mos.head(8), mos.tail(8)])
axes[1].barh([ar(i) for i in ext.index], ext['gap'] * 100,
             color=np.where(ext['gap'] >= 0, '#2a9d8f', '#e76f51'))
axes[1].axvline(0, color='k', lw=0.8)
axes[1].set_title(f'Lowest / highest gap vs expected (mosques with {MIN_EXAMS}+ exams)')
plt.tight_layout()
plt.show()

In [ ]:
tch = vs_expected(g, 'teacher_name', 20)      # low cut-off on purpose, to show the noise

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
axes[0].scatter(tch['exams'], tch['gap'] * 100, s=12, alpha=0.5, color='#3182bd')
axes[0].set_xscale('log')
axes[0].axhline(0, color='k', lw=0.8)
axes[0].set_title('Teacher gap vs number of exams')
axes[0].set_xlabel('exams (log scale)'); axes[0].set_ylabel('pp vs expected')

halaqat = g.dropna(subset=['halaqa_name']).groupby('teacher_name')['halaqa_name'].nunique()
sns.countplot(x=halaqat.clip(upper=5), color='#9e9ac8', ax=axes[1])
axes[1].set_title('Halaqat per teacher (5 = 5 or more)')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

big = tch[tch['exams'] >= MIN_EXAMS]
print(f'{len(big)} teachers have {MIN_EXAMS}+ exams.')
print('\nLowest gap:');  display(big.head(8))
print('Highest gap:'); display(big.tail(8))

The scatter is a funnel: teachers with few exams swing widely, those with many exams stay close to zero. That is why I only rank people above `MIN_EXAMS`, and why one bad year for a small halaqa should not be read as a pattern.

## 8. Students: who fails and comes back

In [ ]:
att = (g.sort_values('exam_date')
         .groupby(['student_key', 'exam_branch'])
         .agg(tries=('passed', 'size'), first_passed=('passed', 'first'), ever_passed=('passed', 'max'))
         .reset_index())

first_fail = att[att['first_passed'] == 0]
retake_success = first_fail['ever_passed'].mean()

per_student = g.groupby('student_key').agg(exams=('passed', 'size'), branches=('exam_branch', 'nunique'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.countplot(x=att['tries'].clip(upper=5), color='#6baed6', ax=axes[0])
axes[0].set_title('Attempts per student per branch (5 = 5 or more)'); axes[0].set_xlabel('')
sns.countplot(x=per_student['branches'].clip(upper=8), color='#74c476', ax=axes[1])
axes[1].set_title('Different branches taken per student (8 = 8 or more)'); axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

print(f'Students who failed a branch on the first try: {len(first_fail):,}')
print(f'...of those, later passed the same branch: {retake_success:.1%}')

## 9. Seasonality

Hijri months are used here because exam seasons follow the Hijri and school calendar, not the Gregorian one.

In [ ]:
HIJRI = ['Muharram', 'Safar', 'Rabi I', 'Rabi II', 'Jumada I', 'Jumada II',
         'Rajab', "Sha'ban", 'Ramadan', 'Shawwal', "Dhu al-Qa'dah", 'Dhu al-Hijjah']

hm = summarize(g.dropna(subset=['h_month']).assign(h_month=lambda x: x['h_month'].astype(int)), 'h_month')
hm = hm.reindex(range(1, 13))

dow = summarize(g.assign(dow=g['exam_date'].dt.day_name()), 'dow')
dow = dow.reindex(['Saturday', 'Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'])

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
axes[0].bar(HIJRI, hm['exams'], color='#9ecae1'); axes[0].tick_params(axis='x', rotation=60)
axes[0].set_title('Exams by Hijri month')
axes[1].plot(HIJRI, hm['pass_rate'] * 100, marker='o', color='#d62728'); axes[1].tick_params(axis='x', rotation=60)
axes[1].set_title('Pass rate % by Hijri month')
axes[2].bar(dow.index, dow['exams'], color='#bcbddc'); axes[2].tick_params(axis='x', rotation=45)
axes[2].set_title('Exams by weekday')
plt.tight_layout()
plt.show()

## 10. Did COVID change anything?

Three windows: before the suspension, the suspension itself, and after in-person classes returned (dates set in the settings cell). The periods have different lengths, so I compare **exams per month**, not totals. And because a different branch mix could change the pass rate on its own, I also fit a small logistic regression that holds branch and sector constant.

In [ ]:
cmp = g[g['exam_date'] >= COMPARE_FROM].copy()
cmp['period'] = np.select([cmp['exam_date'] < COVID_START, cmp['exam_date'] < COVID_END],
                          ['1_pre', '2_covid'], default='3_post')

bounds = {'1_pre': (COMPARE_FROM, COVID_START), '2_covid': (COVID_START, COVID_END),
          '3_post': (COVID_END, cmp['exam_date'].max())}

per = summarize(cmp, 'period')
per['months'] = [(bounds[p][1] - bounds[p][0]).days / 30.44 for p in per.index]
per['exams_per_month'] = per['exams'] / per['months']
per

In [ ]:
s = cmp.set_index('exam_date')['passed'].astype(float)
mon = pd.DataFrame({'exams': s.resample('MS').size(), 'pass_rate': s.resample('MS').mean()})
mon.loc[mon['exams'] == 0, 'pass_rate'] = np.nan

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(mon.index, mon['exams'], color='#3182bd')
axes[0].set_title('Exams per month')
axes[1].plot(mon.index, mon['pass_rate'] * 100, color='#d62728')
axes[1].set_title('Pass rate % per month')
for ax in axes:
    ax.axvspan(COVID_START, COVID_END, color='grey', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
lines = cmp[cmp['exam_year'].between(2018, 2023)]
yoy = lines.groupby(['exam_year', 'exam_month']).size().reset_index(name='exams')

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=yoy, x='exam_month', y='exams', hue='exam_year', palette='viridis', marker='o', ax=ax)
ax.set_xticks(range(1, 13))
ax.set_title('Same months, different years (2020 and 2021 stand out if COVID mattered)')
plt.show()

hp = cmp.pivot_table(index='exam_branch', columns='period', values='passed', aggfunc='mean') * 100
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(hp, annot=True, fmt='.0f', cmap='RdYlGn', center=hp.stack().median(), ax=ax)
ax.set_title('Pass rate % by branch and period'); ax.set_ylabel('branch')
plt.show()

In [ ]:
# is the pass-rate difference bigger than chance? (raw, no adjustment)
ct = pd.crosstab(cmp['period'], cmp['passed'])
chi2, p, _, _ = stats.chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f'chi-square p = {p:.2g}, Cramer V = {cramers_v:.3f}  (V under ~0.1 is a very small effect)')

# same question, holding branch and sector fixed
try:
    import statsmodels.formula.api as smf
    samp = cmp.sample(min(len(cmp), 150_000), random_state=1)
    m = smf.logit('passed ~ C(period) + C(exam_branch) + C(sector_name)', data=samp).fit(disp=0)
    ors = np.exp(pd.concat([m.params, m.conf_int()], axis=1))
    ors.columns = ['odds_ratio', 'ci_low', 'ci_high']
    ors['p_value'] = m.pvalues
    display(ors[ors.index.str.contains('period')])
    print('odds ratio is relative to the pre-COVID period; 1.0 = no change')
except ImportError:
    print('statsmodels is not installed, skipped the adjusted model')

## 11. What the data says

These lines are generated from the tables above, so they update if the data or settings change. I would still read them next to the charts before quoting any of them.

In [ ]:
out = []
out.append(f'1. Volume: {len(g):,} graded exams in {ANALYSIS_START}-{ANALYSIS_END}. '
           f'Busiest year was {int(yr["exams"].idxmax())} with {int(yr["exams"].max()):,} exams.')

grow = yr['exams'].iloc[-3:].mean() / yr['exams'].iloc[:3].mean()
out.append(f'   Average of the last 3 years is {grow:.1f}x the first 3 years in the window.')

big_br = br[br['exams'] >= MIN_EXAMS]
out.append(f'2. Branches: overall pass rate {g["passed"].mean():.1%}, excellent {g["is_excellent"].mean():.1%}. '
           f'Lowest pass rate is branch {int(big_br["pass_rate"].idxmin())} ({big_br["pass_rate"].min():.1%}), '
           f'highest is branch {int(big_br["pass_rate"].idxmax())} ({big_br["pass_rate"].max():.1%}).')

out.append(f'3. Sectors: biggest by volume is {sec["exams"].idxmax()}. After adjusting for branch mix the gap ranges from '
           f'{sec["gap"].min() * 100:+.1f} to {sec["gap"].max() * 100:+.1f} points.')

out.append(f'4. Mosques/teachers: {int((mos["gap"].abs() > 0.05).sum())} of {len(mos)} mosques with {MIN_EXAMS}+ exams are more than '
           f'5 points away from expected; for teachers {int((big["gap"].abs() > 0.05).sum())} of {len(big)}.')

out.append(f'5. Retakes: {retake_success:.1%} of students who failed a branch first time passed it later.')

peak = hm['exams'].idxmax()
out.append(f'6. Seasonality: most exams happen in {HIJRI[int(peak) - 1]}; '
           f'pass rate by Hijri month moves between {hm["pass_rate"].min():.1%} and {hm["pass_rate"].max():.1%}.')

try:
    pre, cov, post = per.loc['1_pre'], per.loc['2_covid'], per.loc['3_post']
    out.append(f'7. COVID: exams per month went from {pre["exams_per_month"]:,.0f} before, to {cov["exams_per_month"]:,.0f} during '
               f'({cov["exams_per_month"] / pre["exams_per_month"] - 1:+.0%}), to {post["exams_per_month"]:,.0f} after '
               f'({post["exams_per_month"] / pre["exams_per_month"] - 1:+.0%} vs before).')
    out.append(f'   Pass rate: {pre["pass_rate"]:.1%} -> {cov["pass_rate"]:.1%} -> {post["pass_rate"]:.1%} '
               f'(raw chi-square p = {p:.2g}, effect size V = {cramers_v:.3f}).')
except KeyError:
    out.append('7. COVID: one of the three periods has no data, check COVID_START / COVID_END.')

print('\n'.join(out))

### Limits worth remembering

- Students are matched by name because ids are missing in many old rows. Two different students with the same full name would be counted as one, and a misspelled name would split one student in two.
- Missing sector and manager were filled from other columns, so those two are estimates for the filled rows.
- The year 2025 ends in November.
- Pass/fail comes from the grade column only. Whether a degree of 1 on a failed exam means "failed" or "absent" cannot be told from this data.
- The COVID windows are my assumption, and the comparison shows association, not cause.

In [ ]:
d.drop(columns=['student_key']).to_csv('exams_clean.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(clean_log).to_csv('cleaning_log.csv', index=False)
print('saved exams_clean.csv and cleaning_log.csv')